# Week 3 — Strengthening the prediction: Publisher + Cross-Validation

Two changes from `02_modeling.ipynb`, based on what that notebook's robustness checks found:

1. **Cross-validation instead of a single Train/Test split for model comparison.**
   `02_modeling.ipynb` Section 8 showed a single Test split's R² is noisy — removing one row
   (Wii Sports) moved Random Forest's Test R² from 0.19 to 0.33. A single split isn't a
   stable enough estimate to trust on its own. Here, 5-fold cross-validation runs across a
   combined **Development pool** (the old Train + Test, 90% of the data) so every fold gets
   its own train/score split and the reported R² is an average over five of them — far less
   sensitive to which few rows happen to be extreme.

2. **`Publisher` added back as a feature, bucketed.** It was excluded from `01_data_wrangling
   _and_split.ipynb`'s feature set for high cardinality (581 unique values), but publisher
   size/marketing budget is one of the strongest real-world sales drivers (Nintendo, EA,
   Activision titles reliably outsell small publishers) — likely the single biggest missing
   signal so far. Fix: keep the top 20 publishers by frequency (covering 64.5% of rows) and
   bucket everything else into `"Other"`.

**Split discipline:** `Validation` is rebuilt fresh here (same 70/20/10 two-step method, same
`RANDOM_STATE`) because the feature set changed, and it is opened only once, at the very end,
after model selection is finished using cross-validation on the Development pool alone.

In [1]:
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split

RANDOM_STATE = 42
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs("splits_v2", exist_ok=True)

## 1. Rebuild the feature set with `Publisher` included

Same cleaning as `01_data_wrangling_and_split.ipynb` (User_Score → numeric, rescaled to
0–100), same leakage exclusions (no NA/EU/JP/Other_Sales), same high-cardinality exclusions
for `Name`/`Developer` — but `Publisher` is now kept.

In [2]:
DATA_PATH = os.path.join("..", "paper_replication", "video_game_sales_and_ratings.csv")

df = pd.read_csv(DATA_PATH)
df["User_Score"] = pd.to_numeric(df["User_Score"], errors="coerce") * 10

FEATURES = [
    "Year_of_Release",
    "Genre",
    "Platform",
    "Rating",
    "Publisher",
    "Critic_Score",
    "Critic_Count",
    "User_Score",
    "User_Count",
]
TARGET = "Global_Sales"

model_df = df[FEATURES + [TARGET]].dropna().reset_index(drop=True)
model_df["Year_of_Release"] = model_df["Year_of_Release"].astype(int)

print(f"Rows before cleaning: {len(df)}")
print(f"Rows after dropping missing values (incl. Publisher): {len(model_df)}")
print("(01_data_wrangling_and_split.ipynb had 6,826 rows without Publisher in the feature set)")
model_df.head()

Rows before cleaning: 16719
Rows after dropping missing values (incl. Publisher): 6825
(01_data_wrangling_and_split.ipynb had 6,826 rows without Publisher in the feature set)


,Year_of_Release,Genre,Platform,Rating,Publisher,Critic_Score,Critic_Count,User_Score,User_Count,Global_Sales
0,2006,Sports,Wii,E,Nintendo,76.0,51.0,80.0,322.0,82.53
1,2008,Racing,Wii,E,Nintendo,82.0,73.0,83.0,709.0,35.52
2,2009,Sports,Wii,E,Nintendo,80.0,73.0,80.0,192.0,32.77
3,2006,Platform,DS,E,Nintendo,89.0,65.0,85.0,431.0,29.80
4,2006,Misc,Wii,E,Nintendo,58.0,41.0,66.0,129.0,28.92


## 2. Bucket `Publisher`

581 unique publishers is too high-cardinality to one-hot encode directly. Keep the top 20 by
frequency (computed on the full modeling set — this is a data-driven bucketing decision, not
a leakage-prone one, since it only uses `Publisher` frequency, unrelated to `Global_Sales`)
and collapse the rest into `"Other"`.

In [3]:
TOP_N_PUBLISHERS = 20
top_publishers = model_df["Publisher"].value_counts().head(TOP_N_PUBLISHERS).index
coverage = model_df["Publisher"].isin(top_publishers).mean()
print(f"Top {TOP_N_PUBLISHERS} publishers cover {coverage:.1%} of rows")

model_df["Publisher_bucketed"] = model_df["Publisher"].where(
    model_df["Publisher"].isin(top_publishers), "Other"
)
model_df["Publisher_bucketed"].value_counts()

Top 20 publishers cover 76.9% of rows


Publisher_bucketed
Other                                     1574
Electronic Arts                            944
Ubisoft                                    496
Activision                                 492
Sony Computer Entertainment                316
THQ                                        307
Nintendo                                   291
Sega                                       284
Take-Two Interactive                       273
Konami Digital Entertainment               244
Namco Bandai Games                         239
Capcom                                     197
Atari                                      176
Microsoft Game Studios                     141
Tecmo Koei                                 137
Square Enix                                134
Warner Bros. Interactive Entertainment     130
Eidos Interactive                          123
Vivendi Games                              113
Codemasters                                111
Midway Games                             

## 3. Train / Test / Validation split (v2)

Same 70/20/10 two-step method as `01_data_wrangling_and_split.ipynb`, rebuilt here because
the underlying `model_df` changed (added Publisher). Saved separately to `splits_v2/` so the
original splits from `01_data_wrangling_and_split.ipynb` stay untouched.

`Test` and `Train` are combined immediately into one **Development** pool for cross-validation
— the 70/20 split point doesn't matter for CV, only the fact that 10% is held out as
Validation.

In [4]:
TRAIN_FRAC, TEST_FRAC, VAL_FRAC = 0.70, 0.20, 0.10

train_test_df, validation_df = train_test_split(
    model_df, test_size=VAL_FRAC, random_state=RANDOM_STATE
)
relative_test_frac = TEST_FRAC / (TRAIN_FRAC + TEST_FRAC)
train_df, test_df = train_test_split(
    train_test_df, test_size=relative_test_frac, random_state=RANDOM_STATE
)

development_df = train_test_df.reset_index(drop=True)  # Train + Test combined, for CV
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)

for name, d in [("Development (Train+Test)", development_df), ("Validation", validation_df)]:
    print(f"{name}: {len(d)} rows ({len(d) / len(model_df):.1%})")

development_df.to_csv("splits_v2/development.csv", index=False)
train_df.to_csv("splits_v2/train.csv", index=False)
test_df.to_csv("splits_v2/test.csv", index=False)
validation_df.to_csv("splits_v2/validation.csv", index=False)

Development (Train+Test): 6142 rows (90.0%)
Validation: 683 rows (10.0%)


## 4. Encode features

One-hot encode `Genre`, `Platform`, `Rating`, `Publisher_bucketed`. Dummy columns are fit on
**Development** only, and Validation is re-indexed onto that same column set.

In [5]:
NUMERIC_FEATURES = ["Year_of_Release", "Critic_Score", "Critic_Count", "User_Score", "User_Count"]
CATEGORICAL_FEATURES = ["Genre", "Platform", "Rating", "Publisher_bucketed"]


def encode_features(d: pd.DataFrame) -> pd.DataFrame:
    cat = pd.get_dummies(d[CATEGORICAL_FEATURES], drop_first=True)
    return pd.concat([d[NUMERIC_FEATURES].reset_index(drop=True), cat.reset_index(drop=True)], axis=1)


X_dev = encode_features(development_df)
X_validation = encode_features(validation_df).reindex(columns=X_dev.columns, fill_value=0)

y_dev = development_df[TARGET]
y_validation = validation_df[TARGET]

print(f"Feature matrix: {X_dev.shape[1]} columns ({len(CATEGORICAL_FEATURES)} categorical sources one-hot encoded)")

Feature matrix: 58 columns (4 categorical sources one-hot encoded)

## 5. 5-fold cross-validation on the Development pool

This replaces the single Train-fit/Test-score comparison from `02_modeling.ipynb`. Each fold
fits on 4/5 of Development and scores on the held-out 1/5, so the reported R² is a mean over
five independent train/score splits — much less sensitive to any single extreme row than one
fixed 80/20 split was.

In [6]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

cv_results = []
for name, model in models.items():
    scores = cross_validate(
        model, X_dev, y_dev, cv=cv,
        scoring=["r2", "neg_root_mean_squared_error", "neg_mean_absolute_error"],
    )
    cv_results.append({
        "model": name,
        "R2_mean": scores["test_r2"].mean(),
        "R2_std": scores["test_r2"].std(),
        "RMSE_mean": -scores["test_neg_root_mean_squared_error"].mean(),
        "MAE_mean": -scores["test_neg_mean_absolute_error"].mean(),
    })

cv_summary = pd.DataFrame(cv_results)[["model", "R2_mean", "R2_std", "RMSE_mean", "MAE_mean"]]
cv_summary.to_csv(os.path.join(OUT_DIR, "cv_metrics_summary_with_publisher.csv"), index=False)
cv_summary

,model,R2_mean,R2_std,RMSE_mean,MAE_mean
0,Linear Regression,0.228953,0.058128,1.709668,0.711601
1,Random Forest,0.437568,0.070867,1.467474,0.508764


## 6. Did this actually help?

Comparison against `02_modeling.ipynb`'s raw-target, no-Publisher, single-Test-split numbers
(Linear Regression R²=0.113, Random Forest R²=0.193). Not a perfectly apples-to-apples
comparison — CV mean vs. a single split — but CV's ±1 std band shows how much of any
difference is real signal vs. split-to-split noise.

In [7]:
comparison = cv_summary.copy()
comparison["R2_no_publisher_single_test_split"] = [0.112597, 0.193194]  # from 02_modeling.ipynb Section 3
comparison["R2_range (mean ± std)"] = comparison.apply(
    lambda r: f"{r['R2_mean']:.3f} ± {r['R2_std']:.3f}", axis=1
)
comparison[["model", "R2_no_publisher_single_test_split", "R2_mean", "R2_range (mean ± std)"]]

,model,R2_no_publisher_single_test_split,R2_mean,R2_range (mean ± std)
0,Linear Regression,0.112597,0.228953,0.229 ± 0.058
1,Random Forest,0.193194,0.437568,0.438 ± 0.071


In [8]:
plt.figure(figsize=(6, 4))
plt.bar(cv_summary["model"], cv_summary["R2_mean"], yerr=cv_summary["R2_std"], capsize=6,
        color=["#4C72B0", "#55A868"])
plt.ylabel("R² (5-fold CV mean ± std, Development pool)")
plt.title("With Publisher (bucketed) + cross-validation")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "cv_r2_with_publisher.png"), dpi=150)
plt.close()
print("Saved outputs/cv_r2_with_publisher.png")

Saved outputs/cv_r2_with_publisher.png


## 7. Final, one-time check on Validation

Refit the better CV model (by mean R²) on the **entire** Development pool, then score once on
Validation — the first and only time Validation is touched in this notebook.

In [9]:
best_model_name = cv_summary.sort_values("R2_mean", ascending=False).iloc[0]["model"]
best_model = models[best_model_name]
print(f"Best model by 5-fold CV mean R2: {best_model_name}")

best_model.fit(X_dev, y_dev)
val_preds = best_model.predict(X_validation)
mse_val = mean_squared_error(y_validation, val_preds)

validation_result = pd.DataFrame([{
    "model": best_model_name,
    "MSE": mse_val,
    "RMSE": np.sqrt(mse_val),
    "MAE": mean_absolute_error(y_validation, val_preds),
    "R2": r2_score(y_validation, val_preds),
}])
validation_result.to_csv(os.path.join(OUT_DIR, "validation_metrics_with_publisher_cv.csv"), index=False)
validation_result

Best model by 5-fold CV mean R2: Random Forest


,model,MSE,RMSE,MAE,R2
0,Random Forest,0.763317,0.87368,0.454204,0.679893


## 8. Does Publisher actually matter? Feature importance

In [10]:
if best_model_name == "Random Forest":
    importances = pd.Series(best_model.feature_importances_, index=X_dev.columns).sort_values(ascending=False)
else:
    importances = pd.Series(best_model.coef_, index=X_dev.columns).sort_values(key=np.abs, ascending=False)

plt.figure(figsize=(8, 7))
importances.head(20)[::-1].plot(kind="barh")
plt.title(f"Top feature importances - {best_model_name} (with Publisher)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "feature_importance_with_publisher.png"), dpi=150)
plt.close()

publisher_features = importances[importances.index.str.startswith("Publisher_bucketed")]
print(f"Publisher-related features in the top 20: {(importances.head(20).index.str.startswith('Publisher_bucketed')).sum()}")
print()
print("Top 15 features overall:")
importances.head(15)

Publisher-related features in the top 20: 4

Top 15 features overall:


User_Count                                   0.275697
Year_of_Release                              0.093196
Critic_Score                                 0.092274
Critic_Count                                 0.084211
Platform_Wii                                 0.081983
Genre_Sports                                 0.077351
User_Score                                   0.050569
Platform_PC                                  0.047282
Rating_E                                     0.036198
Platform_DS                                  0.031818
Publisher_bucketed_Nintendo                  0.027593
Genre_Racing                                 0.018552
Genre_Misc                                   0.013822
Platform_PS2                                 0.010295
Publisher_bucketed_Microsoft Game Studios    0.007589
dtype: float64

## Summary

- **Cross-validation** replaced the single, noisy Train/Test split from `02_modeling.ipynb`
  with a 5-fold mean ± std over the Development pool — a materially more stable estimate of
  how well these features predict `Global_Sales`.
- **Publisher (bucketed to top 20 + Other)** was added as a feature, restoring a real-world
  sales driver that was previously excluded for cardinality reasons.
- Compare the CV R² above against `02_modeling.ipynb`'s no-Publisher, single-split numbers
  (Linear Regression 0.113, Random Forest 0.193) to see how much of any improvement is real
  vs. within CV's own std band — and check the feature-importance table to see whether
  Publisher actually earned a place among the top predictors, or whether Critic/User
  reception still dominates.
- The Validation R² above (touched exactly once, after model selection was finished using CV
  alone) is the number to report as this iteration's headline result.